In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from scipy.stats import norm

# =========================================================
# 1. Data: 6D inputs and 1D outputs (Function 7)
# =========================================================

X_raw = np.array([
    [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
    [0.54300258, 0.9246939 , 0.34156746, 0.64648585, 0.71844033, 0.34313266],
    [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654 ],
    [0.11886697, 0.61505494, 0.90581639, 0.8553003 , 0.41363143, 0.58523563],
    [0.63021764, 0.8380969 , 0.68001305, 0.73189509, 0.52673671, 0.34842921],
    [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
    [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
    [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
    [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164 ],
    [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
    [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
    [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
    [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
    [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
    [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
    [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
    [0.41762629, 0.06409998, 0.24566877, 0.5590408 , 0.19153138, 0.25464092],
    [0.72628566, 0.46489581, 0.92457051, 0.8072454 , 0.6354384 , 0.14341787],
    [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825 ],
    [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
    [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
    [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
    [0.68685257, 0.04101721, 0.00757301, 0.285009  , 0.69156848, 0.6555429 ],
    [0.17597754, 0.6244165 , 0.29554198, 0.46955276, 0.09776977, 0.72814108],
    [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
    [0.06661051, 0.52804507, 0.8160952 , 0.96101714, 0.08650933, 0.77778822],
    [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
    [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983 , 0.08130609],
    [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764 , 0.36288398],
    [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547 , 0.07966402],
    [1.04245 , 1.024693, 1.02457 , 1.061017, 1.098654, 1.051013],
    [0.019976, 0.432955, 0.301662, 0.169496, 0.348651, 0.743371],
    [0.611853, 0.139495, 0.292145, 0.366362, 0.45607 , 0.785175],
    [0.015006, 0.390905, 0.178469, 0.119929, 0.088415, 0.904408],
    [0.046821, 0.309546, 0.608802, 0.064364, 0.39334 , 0.990644],
    [0.011478, 0.62027 , 0.525606, 0.053535, 0.52488 , 0.666127],
    [0.028679, 0.235471, 0.148723, 0.076614, 0.11285 , 0.837107]
])

y_raw = np.array([
    6.04432696e-01, 5.62753067e-01, 7.50323668e-03, 6.14243025e-02,
    2.73046801e-01, 8.37465723e-02, 1.36496830e+00, 9.26449549e-02,
    1.78695987e-02, 3.35649360e-02, 7.35163042e-02, 2.06309698e-01,
    8.82563400e-03, 2.68400317e-01, 6.11525528e-01, 1.47981826e-02,
    2.74892508e-01, 6.67632469e-02, 4.21183545e-02, 2.70146502e-03,
    1.82090730e-02, 7.01602756e-03, 1.00506611e-01, 4.75395516e-01,
    6.75141631e-01, 5.16457219e-01, 3.77747962e-03, 3.13433331e-03,
    2.13425228e-02, 9.54111589e-02, 4.636858051500375e-06, 1.680828424430851,
    1.1170576710554418, 0.44099891630237703, 0.8950628737420184,
    0.664856997347448, 0.6583383225997628
])

# =========================================================
# 2. Configuration
# =========================================================

RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

INPUT_DIM = X_raw.shape[1]
N_CANDIDATES_FINAL = 20000
EI_XI = 0.01

# =========================================================
# 3. Surrogate model
# =========================================================

class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)

# =========================================================
# 4. Acquisition functions (numerically stable)
# =========================================================

def gaussian_ei(mu, sigma, f_best, xi=0.0):
    sigma = np.maximum(sigma, 1e-6)
    z = (mu - f_best - xi) / sigma
    return (mu - f_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)

# =========================================================
# 5. Main logic
# =========================================================

def main():
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_scaled = x_scaler.fit_transform(X_raw)
    y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

    model = MLPRegressorTorch(INPUT_DIM)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32)

    model.train()
    for _ in range(1500):
        optimizer.zero_grad()
        loss = loss_fn(model(X_tensor), y_tensor)
        loss.backward()
        optimizer.step()

    # Candidate generation
    rng = np.random.RandomState(RANDOM_STATE)
    X_cand = rng.uniform(0, 1, size=(N_CANDIDATES_FINAL, INPUT_DIM))
    X_cand_scaled = x_scaler.transform(X_cand)

    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_cand_scaled, dtype=torch.float32)).numpy().ravel()

    mu = y_scaler.inverse_transform(preds.reshape(-1, 1)).ravel()
    sigma = np.std(mu) * np.ones_like(mu)

    f_best = np.max(y_raw)
    ei = gaussian_ei(mu, sigma, f_best, xi=EI_XI)

    # Stable selection near local maxima
    top = np.where(ei >= 0.995 * np.max(ei))[0]
    best_idx = top[np.argmax(mu[top])]

    x_next = np.round(X_cand[best_idx], 6)

    print("RECOMMENDED NEXT POINT")
    print("x_next =", x_next)

if __name__ == "__main__":
    main()


RECOMMENDED NEXT POINT
x_next = [0.069198 0.39455  0.352452 0.093928 0.370707 0.725655]
